# FFAI RAG Pipeline: End-to-End with Real LLM

This notebook demonstrates the full RAG pipeline using **real LLM calls**
through `FFAI.query()`. No mocks.

Pipeline: `Document → Chunk → Index (BM25) → Search → FFAI.query() → Answer`

Key features demonstrated:
- **Auto-wiring**: `FFAI(client, rag=rag)` automatically bridges the LLM client to RAG
- **Rich metadata**: `QueryResult` carries usage, cost, and latency from the LLM call
- **BM25-only**: No ChromaDB dependency — pure keyword search

Requires a `MISTRAL_API_KEY` (or other provider key) in your environment.

## Section 1: Setup

In [ ]:
import sys
from pathlib import Path

_cwd = Path().resolve()
_project_root = _cwd
for _p in [_cwd, *list(_cwd.parents)]:
    if (_p / 'pyproject.toml').is_file():
        _project_root = _p
        break

if str(_project_root) not in sys.path:
    sys.path.insert(0, str(_project_root))

from ffai.Clients.FFLiteLLMClient import FFLiteLLMClient  # noqa: E402
from ffai.FFAI import FFAI  # noqa: E402
from ffai.rag import RAG  # noqa: E402

print(f'Project root: {_project_root}')
print('Imports successful')

<div class="page-break"></div>

---

## Section 2: Create LLM Client and RAG

`RAG(bm25_alpha=0.6)` enables BM25 keyword search without requiring ChromaDB.

Passing `rag=rag` to `FFAI()` **auto-wires** the client: internally, `FFAI` creates
a `ClientAdapter` that wraps `client.generate_response()` and sets it as RAG's
default `generate_fn`. This means `ffai.query()` works without any manual wiring.

In [ ]:
client = FFLiteLLMClient(
    model_string='mistral/mistral-small-2503',
    temperature=0.3,
    max_tokens=512,
    system_instructions='You are a helpful assistant. Answer concisely based on the provided context.',
)

rag = RAG(
    embed='mistral/mistral-embed',
    bm25_alpha=0.6,
    chunker='markdown',
    chunk_size=500,
    chunk_overlap=50,
)

# Auto-wire: FFAI sets ClientAdapter(client) as rag's default generate_fn
ffai = FFAI(client=client, rag=rag)

print(f'Client: {client}')
print(f'RAG: bm25_alpha={rag._bm25_alpha}, store={rag._store is not None}')
print(f'Auto-wired generate_fn: {rag._generate_fn is not None}')
print(f'FFAI ready: rag={ffai.rag is not None}')

<div class="page-break"></div>

---

## Section 3: Load and Index Documents

Download sample documents and index them into BM25. Each call to `rag.index()`
chunks the text and adds it to the BM25 index.

In [ ]:
from examples._rag_data.download import download

docs = download()
for name, path in docs.items():
    print(f'  {name}: {path.stat().st_size:,} bytes')

In [ ]:
python_tutorial = docs['python_tutorial.md'].read_text(encoding='utf-8')
api_docs = docs['api_docs.md'].read_text(encoding='utf-8')

n1 = rag.index(python_tutorial, source='python_tutorial.md')
n2 = rag.index(api_docs, source='api_docs.md')

print(f'Indexed {n1} chunks from python_tutorial.md')
print(f'Indexed {n2} chunks from api_docs.md')
print(f'Total chunks in BM25: {rag.count()}')

<div class="page-break"></div>

---

## Section 4: Inspect Chunks

Preview what the chunker produced.

In [ ]:
chunks = rag.chunk(python_tutorial, source='python_tutorial.md')

print(f'Total chunks: {len(chunks)}')
print()
for i, chunk in enumerate(chunks[:3]):
    preview = chunk.content[:100].replace(chr(10), ' ')
    print(f'  Chunk {i}: chars={chunk.start_char}-{chunk.end_char} | {preview}...')

<div class="page-break"></div>

---

## Section 5: Search (Retrieval Only)

Use `rag.search()` to retrieve relevant chunks without generating an answer.

In [ ]:
hits = rag.search('async programming', top_k=3)

print(f'Top {len(hits)} results for "async programming":')
print()
for i, hit in enumerate(hits):
    preview = hit.content[:120].replace(chr(10), ' ')
    print(f'  [{i+1}] score={hit.score:.4f} source={hit.source}')
    print(f'      {preview}...')
    print()

<div class="page-break"></div>

---

## Section 6: End-to-End Query with Metadata

One call: search → format context → prompt → LLM → answer.

`FFAI.query()` returns a `QueryResult` with rich metadata from the LLM call:
- `usage` — token counts from the provider
- `cost_usd` — estimated cost in USD
- `duration_ms` — wall-clock latency of the generation call

In [ ]:
result = ffai.query(
    'How do I run multiple async tasks concurrently in Python?',
    top_k=3,
)

print(f'Answer:\n{result.answer}')
print(f'\nSources: {result.sources}')
print(f'Hits: {len(result.hits)}')
print(f'Prompt length: {len(result.prompt):,} chars')
print('\n--- Metadata ---')
print(f'Usage: {result.usage}')
print(f'Cost: ${result.cost_usd:.6f}')
print(f'Duration: {result.duration_ms:.1f} ms' if result.duration_ms else 'Duration: N/A')

<div class="page-break"></div>

---

## Section 7: Multiple Queries with Cost Tracking

Run several questions and track cumulative cost across calls.

In [ ]:
questions = [
    'What is the purpose of asyncio.gather()?',
    'How do I handle timeouts in async code?',
    'What are the rate limits for the login endpoint?',
    'What error format does the API use?',
]

total_cost = 0.0
for q in questions:
    result = ffai.query(q, top_k=3)
    total_cost += result.cost_usd
    answer_preview = result.answer[:150].replace(chr(10), ' ')
    print(f'Q: {q}')
    print(f'A: {answer_preview}...')
    print(f'   Sources: {result.sources}  Cost: ${result.cost_usd:.6f}')
    print()

print(f'Total cost for {len(questions)} queries: ${total_cost:.6f}')

<div class="page-break"></div>

---

## Section 8: Custom Prompt Template

Override the default prompt template for domain-specific formatting.

In [ ]:
custom_template = (
    'You are a Python expert. Using ONLY the context below, answer the question.\n\n'
    'Context:\n{context}\n\n'
    'Question: {question}\n\n'
    'Provide a code example if applicable.'
)

result = ffai.query(
    'How do task groups work?',
    top_k=3,
    prompt_template=custom_template,
)

print(f'Answer:\n{result.answer}')
print(f'\nSources: {result.sources}')
print(f'Cost: ${result.cost_usd:.6f}  Duration: {result.duration_ms:.1f} ms' if result.duration_ms else f'Cost: ${result.cost_usd:.6f}')

<div class="page-break"></div>

---

## Section 9: Inspect the Generated Prompt

See exactly what was sent to the LLM — context + question.

In [ ]:
result = ffai.query('What are async context managers?', top_k=2)

print('=== Full Prompt Sent to LLM ===')
print(result.prompt)
print('=== End Prompt ===')
print(f'\nAnswer: {result.answer[:200]}...')
print(f'Usage: {result.usage}')

<div class="page-break"></div>

---

## Section 10: FFAI Facade Methods

The FFAI object now exposes `index()`, `search()`, `delete()`, and `count()` directly
— no need to reach through to `ffai.rag`.

In [ ]:
# Index through the facade
n1 = ffai.index(python_tutorial, source='python_tutorial.md')
n2 = ffai.index(api_docs, source='api_docs.md')
print(f'Indexed {n1} + {n2} = {ffai.count()} chunks via facade')

In [ ]:
# Search through the facade (no LLM call)
hits = ffai.search('async programming', top_k=3)
print(f'Found {len(hits)} hits:')
for h in hits:
    print(f'  score={h.score:.4f} source={h.source}')

<div class="page-break"></div>

---

## Section 11: Cost-Saving Features

### `allow_llm_on_empty=False`

When no search hits are found, skip the LLM call entirely. Saves tokens and money.

### `generate_timeout`

Cap how long the LLM generation can take. Raises `TimeoutError` if exceeded.
Note: the underlying API request may still complete (threads can't be cancelled),
so cost may still be incurred.

In [ ]:
result = ffai.query(
    'quantum computing entanglement protocols',
    allow_llm_on_empty=False,
)
print(f'Answer: "{result.answer}"')
print(f'Hits: {len(result.hits)}')
print(f'Sources: {result.sources}')
print(f'Cost: ${result.cost_usd:.6f} (should be $0 — no LLM call)')

In [ ]:
result = ffai.query(
    'What is asyncio.gather()?',
    top_k=3,
    generate_timeout=30.0,
)
print(f'Answer: {result.answer[:150]}...')
print(f'Duration: {result.duration_ms:.1f} ms' if result.duration_ms else 'Duration: N/A')

<div class="page-break"></div>

---

## Section 12: Direct `RAG.query()` with Custom `generate_fn`

You can also call `rag.query()` directly with a custom generation function.
If it returns a `GenerationResult`, the metadata (usage, cost, duration)
flows through to the `QueryResult`.

In [ ]:
from ffai.rag.types import GenerationResult


def custom_generate(prompt: str) -> GenerationResult:
    return GenerationResult(
        text='Python async context managers use __aenter__ and __aexit__.',
        usage={'prompt_tokens': 50, 'completion_tokens': 15, 'total_tokens': 65},
        cost_usd=0.00008,
        duration_ms=120.0,
    )

result = rag.query('What are async context managers?', generate_fn=custom_generate)

print(f'Answer: {result.answer}')
print(f'Usage: {result.usage}')
print(f'Cost: ${result.cost_usd:.6f}')
print(f'Duration: {result.duration_ms:.1f} ms')
print(f'Sources: {result.sources}')

<div class="page-break"></div>

---

## Section 13: Filtered Search by Source

Pass `source=...` to limit retrieval to a specific document.

In [ ]:
result = ffai.query(
    'What are the rate limits?',
    top_k=3,
    source='api_docs.md',
)

print(f'Answer: {result.answer[:200]}...')
print(f'Sources: {result.sources}')
print(f'All hits from api_docs.md: {all(h.source == "api_docs.md" for h in result.hits)}')

<div class="page-break"></div>

---

## Section 14: Summary

In [ ]:
import polars as pl

summary = pl.DataFrame([
    {'Step': 1, 'Component': 'Client', 'API': 'FFLiteLLMClient(model_string=...)', 'Role': 'LLM provider'},
    {'Step': 2, 'Component': 'RAG', 'API': 'RAG(bm25_alpha=0.6)', 'Role': 'Chunk + index + retrieve'},
    {'Step': 3, 'Component': 'FFAI', 'API': 'FFAI(client, rag=rag)', 'Role': 'Auto-wire client to RAG'},
    {'Step': 4, 'Component': 'Facade', 'API': 'ffai.index/search/delete/count', 'Role': 'Manage RAG via FFAI'},
    {'Step': 5, 'Component': 'Query', 'API': 'ffai.query(question, top_k=3)', 'Role': 'Search + generate + metadata'},
    {'Step': 6, 'Component': 'Metadata', 'API': 'result.usage / .cost_usd / .duration_ms', 'Role': 'Token counts, cost, latency'},
    {'Step': 7, 'Component': 'Cost Control', 'API': 'allow_llm_on_empty=False', 'Role': 'Skip LLM when no context'},
    {'Step': 8, 'Component': 'Timeout', 'API': 'generate_timeout=30.0', 'Role': 'Cap LLM call duration'},
])

print(summary)